# JupyterHub Mailout Notebook

This notebook fetches resource and user information from a JSON API and sends notifications.

This notebook uses _getpass_ to prompt you to paste in your authentication tokens.

In [ ]:
import datetime
import requests
from getpass import getpass
from mailout_helper import MailoutHelper

# JupyterHub API config
*NOTE*: To obtain an API Token, visit: 
* https://jupyterhub.rc.nectar.org.au/hub/token OR
* https://binder.rc.nectar.org.au/hub/token (BinderHub)

In [ ]:
JUPYTERHUB_API_URL = 'https://jupyterhub.rc.nectar.org.au/hub/api'
#JUPYTERHUB_API_URL = 'https://binder.rc.nectar.org.au/hub/api'
JUPYTERHUB_API_TOKEN = getpass(prompt='Enter your JupyterHub API Token: ')
SIX_MONTHS_IN_DAYS = 180

## Set up the Mailout Helper

Create a Mailout Helper instance

In [ ]:
helper = MailoutHelper()

## Set start/end times for a scheduled outage

This is *optional* if your mailout is for an outage

In [ ]:
#helper.set_times(
#    start_time = '2026-03-01 09:00:00',
#    end_time = '2026-03-01 17:00:00',
#    timezone = 'Australia/Melbourne',
#)

## Setup OpenStack

The OpenStack API token here is used for authentication to Taynac for notification delivery. You will need to use credentials that have admin.

In [ ]:
helper.setup_openstack('https://identity.rc.nectar.org.au/')

## JupyterHub API access

This helper code implements access to the JupyterHub API and can fetch users

In [ ]:
def _jupyterhub_api(path, *, limit=200, order_by="last_activity", order="desc"):
    """
    Fetch all items from a paginated JupyterHub list endpoint,
    sorted by last_activity (descending by default).
    """
    url = JUPYTERHUB_API_URL.rstrip("/") + path
    headers = {
        "Accept": "application/jupyterhub-pagination+json",
        "Authorization": f"token {JUPYTERHUB_API_TOKEN}",
    }

    params = {
        "limit": limit,
        "offset": 0,
        "order_by": order_by,
        "order": order,
    }

    items = []
    while True:
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        data = r.json()

        items.extend(data["items"])

        nxt = data["_pagination"]["next"]
        if not nxt:
            break

        params["offset"] = nxt.get("offset", params["offset"] + params["limit"])
        params["limit"] = nxt.get("limit", params["limit"])
        url = nxt.get("url", url)

    return items


def get_recent_users():
    """Return only users with last_activity within the past `months` months."""
    cutoff = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=SIX_MONTHS_IN_DAYS)
    users = _jupyterhub_api("/users")
    recent_users = [
        u for u in users
        if u.get("last_activity") and datetime.datetime.fromisoformat(u['last_activity']) >= cutoff
    ]
    return recent_users

## Fetch the JupyterHub users and extract their email addresses

In [ ]:
hubusers = get_recent_users()
emails = [u['name'] for u in hubusers]
print(f"Found {len(emails)} email addresses")

## Define Subject and Body

In [ ]:
# Subject for notfications
subject = 'Important notice for ARDC Nectar Jupyter Notebook Service upgrade'

# Body
body = """<p>Dear ARDC Nectar Jupyter Notebook Service User,</p>
<br />
<p>This email is to inform you that the ARDC Nectar Jupyter Notebook Service 
has recently been upgraded, and this may affect your existing workspace.</p>
<br />
<p>
As part of this upgrade the Jupyter environments have also been upgraded to
include the latest versions of all packages, and most notably, the installed Python
version, which has been upgraded from Python 3.10 to 3.13.
</p>
<br />
<p>If you have installed extra packages in your Python or R environments,
you may need to install them again to match the new Python or R version
now available.</p>
<br />
<p>There might also be some Python language changes that might cause
incompatibilities with existing code. Consider browsing the 
<a href="https://docs.python.org/3/whatsnew/index.html">Python What's New</a>
website for information about changes in recent Python versions.</p>
<br />
<p>If you have any questions regarding this notice, please contact us by
replying to this email or by emailing <a href="mailto:support@ehelp.edu.au">support@ehelp.edu.au</a>.</p>
<br />
<p>Regards,
<p>ARDC Nectar Research Cloud Support Team</p>
<br />
<p><small><i>This email has been sent to you because you have used the ARDC Nectar Jupyter Notebook service within the last six months.
These emails are essential communications which we endeavour to keep to a minimum.</i></small></p>
"""

## Generate and Send Notifications

In [ ]:
notifications = []

context = helper.build_context()
rendered_body = helper.render_template_string(body, context)
rendered_subject = helper.render_template_string(subject, context)

for email in emails:
    notifications.append({
        'subject': rendered_subject,
        'body': rendered_body,
        'to': email,
    })
print(f"Found {len(notifications)} notifications")

## Preview Notifications

In [ ]:
# Preview first notification
helper.preview_notification(notifications[0])

## Send Notifications

In [ ]:
# Are you ready to send these?
for notification in notifications:
    print(f"Sending notification to {notification['to']}...")
    #helper.send_notification(notification)

print(f"Finished")